In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """多头注意力机制"""
    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.d_k = d_model // nhead
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.out_linear = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # 线性变换并分头
        q = self.q_linear(q).view(batch_size, -1, self.nhead, self.d_k).transpose(1,2)
        k = self.k_linear(k).view(batch_size, -1, self.nhead, self.d_k).transpose(1,2)
        v = self.v_linear(v).view(batch_size, -1, self.nhead, self.d_k).transpose(1,2)
        
        # 计算注意力
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        # 合并多头
        output = torch.matmul(attn, v)
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.out_linear(output)

class FeedForward(nn.Module):
    """前馈网络"""
    def __init__(self, d_model, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        
    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = self.dropout(x)
        return self.linear2(x)

class EncoderLayer(nn.Module):
    """编码器层"""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.ffn = FeedForward(d_model, dim_feedforward, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, src, src_mask=None):
        # 自注意力
        src2 = self.self_attn(src, src, src, src_mask)
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        
        # 前馈网络
        src2 = self.ffn(src)
        src = src + self.dropout2(src2)
        return self.norm2(src)

class DecoderLayer(nn.Module):
    """解码器层"""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.cross_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.ffn = FeedForward(d_model, dim_feedforward, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        # 自注意力（带掩码）
        tgt2 = self.self_attn(tgt, tgt, tgt, tgt_mask)
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        
        # 交叉注意力（编码器输出作为k,v）
        tgt2 = self.cross_attn(tgt, memory, memory, memory_mask)
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        
        # 前馈网络
        tgt2 = self.ffn(tgt)
        tgt = tgt + self.dropout3(tgt2)
        return self.norm3(tgt)

class TransformerEncoder(nn.Module):
    """完整编码器"""
    def __init__(self, num_layers, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        
    def forward(self, src, src_mask=None):
        for layer in self.layers:
            src = layer(src, src_mask)
        return src

class TransformerDecoder(nn.Module):
    """完整解码器"""
    def __init__(self, num_layers, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        for layer in self.layers:
            tgt = layer(tgt, memory, tgt_mask, memory_mask)
        return tgt

In [2]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import argparse
import os
from tqdm import tqdm
from transformers import BertTokenizerFast

class Config:
    def __init__(self):
        self.batch_size = 32
        self.learning_rate = 3e-5
        self.epochs = 10
        self.max_length = 256
        self.model_dir = "/kaggle/working"
        self.train_path = "/kaggle/input/v1-1-json/train-v1.1.json"
        self.dev_path = "/kaggle/input/v1-1-json/dev-v1.1.json"
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.d_model = 768
        self.nhead = 12
        self.dim_feedforward = 3072
        self.dropout = 0.1
        self.num_layers = 7

class SQuADProcessor:
    def __init__(self, config):
        self.config = config
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
    
    def load_data(self, path):
        with open(path, 'r') as f:
            return json.load(f)['data']
    
    def process(self, data):
        examples = []
        for article in data:
            for para in article['paragraphs']:
                context = para['context']
                for qa in para['qas']:
                    example = {
                        'context': context,
                        'question': qa['question'],
                        'answer': qa['answers'][0]
                    }
                    examples.append(example)
        return examples
    
    def create_features(self, examples):
        input_ids, masks = [], []
        start_pos, end_pos = [], []
        
        for ex in examples:
            encoding = self.tokenizer(
                ex['question'],
                ex['context'],
                max_length=self.config.max_length,
                truncation=True,
                padding='max_length',
                return_offsets_mapping=True
            )
            
            ans_start = ex['answer']['answer_start']
            ans_end = ans_start + len(ex['answer']['text'])
            
            start_char = ans_start
            end_char = ans_end
            sequence_ids = encoding.sequence_ids()
            
            # Find token positions
            start_token, end_token = -1, -1
            for i, (idx, (s, e)) in enumerate(zip(sequence_ids, encoding.offset_mapping)):
                if idx != 1: continue  # Only look at context tokens
                if s <= start_char < e: start_token = i
                if s < end_char <= e: end_token = i
            
            if start_token != -1 and end_token != -1:
                input_ids.append(encoding['input_ids'])
                masks.append(encoding['attention_mask'])
                start_pos.append(start_token)
                end_pos.append(end_token)
        
        return {
            'input_ids': torch.tensor(input_ids),
            'attention_mask': torch.tensor(masks),
            'start_pos': torch.tensor(start_pos),
            'end_pos': torch.tensor(end_pos)
        }

class SQuADDataset(Dataset):
    def __init__(self, features):
        self.features = features
    
    def __len__(self):
        return len(self.features['input_ids'])
    
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.features.items()}

class TransformerQA(nn.Module):
    def __init__(self, config, tokenizer):
        super().__init__()
        self.tokenizer = tokenizer 
        self.embedding = nn.Embedding(self.tokenizer.vocab_size, config.d_model)
        self.pos_encoder = nn.Parameter(torch.zeros(config.max_length, config.d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.nhead,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, config.num_layers)
        self.start_fc = nn.Linear(config.d_model, 1)
        self.end_fc = nn.Linear(config.d_model, 1)
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids) + self.pos_encoder[:input_ids.size(1)]
        x = self.dropout(x)
        x = x.permute(1, 0, 2)  # [seq_len, bs, dim]
        
        output = self.encoder(x, src_key_padding_mask=~attention_mask.bool())
        output = output.permute(1, 0, 2)  # [bs, seq_len, dim]
        
        start_logits = self.start_fc(output).squeeze(-1)
        end_logits = self.end_fc(output).squeeze(-1)
        return start_logits, end_logits

class QATrainer:
    def __init__(self, config, model, train_loader, dev_loader=None):
        self.config = config
        self.model = model.to(config.device)
        self.train_loader = train_loader
        self.dev_loader = dev_loader
        self.optimizer = AdamW(model.parameters(), lr=config.learning_rate)
    
    def train_epoch(self):
        self.model.train()
        total_loss = 0
        for batch in tqdm(self.train_loader, desc="Training"):
            input_ids = batch['input_ids'].to(self.config.device)
            mask = batch['attention_mask'].to(self.config.device)
            start = batch['start_pos'].to(self.config.device)
            end = batch['end_pos'].to(self.config.device)
            
            self.optimizer.zero_grad()
            s_logits, e_logits = self.model(input_ids, mask)
            
            loss = nn.CrossEntropyLoss()(s_logits, start) + \
                   nn.CrossEntropyLoss()(e_logits, end)
            
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        return total_loss / len(self.train_loader)
    
    def evaluate(self):
        self.model.eval()
        # 请补全evaluate过程
        total_loss = 0
        start_correct = 0
        end_correct = 0
        exact_match = 0
        total_samples = 0
        
        with torch.no_grad():
            for batch in tqdm(self.dev_loader, desc="Evaluating"):
                input_ids = batch['input_ids'].to(self.config.device)
                mask = batch['attention_mask'].to(self.config.device)
                start_pos = batch['start_pos'].to(self.config.device)
                end_pos = batch['end_pos'].to(self.config.device)
                
                # 获取模型预测
                s_logits, e_logits = self.model(input_ids, mask)
                
                # 计算损失
                loss = nn.CrossEntropyLoss()(s_logits, start_pos) + \
                       nn.CrossEntropyLoss()(e_logits, end_pos)
                total_loss += loss.item()
                
                # 获取预测位置
                pred_start = torch.argmax(s_logits, dim=1)
                pred_end = torch.argmax(e_logits, dim=1)
                
                # 计算指标
                start_correct += (pred_start == start_pos).sum().item()
                end_correct += (pred_end == end_pos).sum().item()
                exact_match += ((pred_start == start_pos) & (pred_end == end_pos)).sum().item()
                total_samples += len(input_ids)
        
        # 计算各项指标
        metrics = {
            'loss': total_loss / len(self.dev_loader),
            'start_acc': start_correct / total_samples,
            'end_acc': end_correct / total_samples,
            'exact_match': exact_match / total_samples,
            'avg_acc': (start_correct + end_correct) / (2 * total_samples)
        }
        
        return metrics
    
    def save_model(self, path):
        torch.save(self.model.state_dict(), path)
    
    def load_model(self, path):
        self.model.load_state_dict(torch.load(path))

In [3]:
def run_full_process(model_save_path="/kaggle/working/model.pt"):
    # 初始化配置和处理器
    config = Config()
    processor = SQuADProcessor(config)

    # 训练流程
    print("Starting training process...")
    train_data = processor.load_data(config.train_path)
    train_examples = processor.process(train_data)
    train_features = processor.create_features(train_examples)
    train_dataset = SQuADDataset(train_features)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    
    dev_data = processor.load_data(config.dev_path)
    dev_examples = processor.process(dev_data)
    dev_features = processor.create_features(dev_examples)
    dev_loader = DataLoader(SQuADDataset(dev_features), batch_size=config.batch_size)
    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
    model = TransformerQA(config, tokenizer)  # 传入tokenizer
    #model = TransformerQA(config)
    trainer = QATrainer(config, model, train_loader, dev_loader)
    
    for epoch in range(config.epochs):
        avg_loss = trainer.train_epoch()
        print(f"Epoch {epoch+1} Loss: {avg_loss}")
        metrics = trainer.evaluate()
        print(f"Epoch {epoch+1} Validation Metrics: {metrics}")
        trainer.save_model(f"{config.model_dir}/epoch_{epoch+1}.pt")
    
    # 测试流程（使用最后保存的模型）
    print("\nStarting testing process with the trained model...")
    final_model_path = f"{config.model_dir}/epoch_{config.epochs}.pt"
    
    test_data = processor.load_data(config.dev_path)  # 假设使用dev集作为测试
    test_examples = processor.process(test_data)
    test_features = processor.create_features(test_examples)
    test_loader = DataLoader(SQuADDataset(test_features), batch_size=config.batch_size)
    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
    test_model = TransformerQA(config, tokenizer)
    test_trainer = QATrainer(config, test_model, None, test_loader)
    test_trainer.load_model(final_model_path)
    test_metrics = test_trainer.evaluate()
    print(f"\nFinal Test Results - {test_metrics}")

# 在Jupyter中直接运行
run_full_process()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Starting training process...


...

Epoch 10 Loss: 5.910293908555891


Evaluating: 100%|██████████| 326/326 [01:41<00:00,  3.23it/s]


Epoch 10 Validation Metrics: {'loss': 8.44064412965365, 'start_acc': 0.12064162904620113, 'end_acc': 0.12650081644414563, 'exact_match': 0.06003265776582461, 'avg_acc': 0.12357122274517338}

Starting testing process with the trained model...


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
/tmp/ipykernel_19/1945565372.py:207: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you s


Final Test Results - {'loss': 8.44064412965365, 'start_acc': 0.12064162904620113, 'end_acc': 0.12650081644414563, 'exact_match': 0.06003265776582461, 'avg_acc': 0.12357122274517338}
